In [0]:
storage_account_name = "segrid"
storage_account_key = "STORAGE ACCOUNT KEY"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
connection_string = "PRIMARY CONNECTION STRING"

eh_namespace = "evhns-segrid-anuj-std.servicebus.windows.net:9093"
eh_name = "meter-readings"

kafka_options = {
    "kafka.bootstrap.servers": eh_namespace,
    "subscribe": eh_name,
    "kafka.sasl.mechanism": "PLAIN",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.jaas.config": (
        'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="$ConnectionString" password="{connection_string}";'
    ),
    "startingOffsets": "earliest",
}

df_stream_raw = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

df_stream_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [0]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType

json_schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Time", StringType(), True),
    StructField("Global_active_power", StringType(), True),
    StructField("Global_reactive_power", StringType(), True),
    StructField("Voltage", StringType(), True),
    StructField("Global_intensity", StringType(), True),
    StructField("Sub_metering_1", StringType(), True),
    StructField("Sub_metering_2", StringType(), True),
    StructField("Sub_metering_3", StringType(), True),
    StructField("sent_at", StringType(), True),
])

df_parsed = (
    df_stream_raw
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(from_json(col("json_str"), json_schema).alias("data"))
    .select("data.*")
)

df_parsed.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: string (nullable = true)
 |-- Global_active_power: string (nullable = true)
 |-- Global_reactive_power: string (nullable = true)
 |-- Voltage: string (nullable = true)
 |-- Global_intensity: string (nullable = true)
 |-- Sub_metering_1: string (nullable = true)
 |-- Sub_metering_2: string (nullable = true)
 |-- Sub_metering_3: string (nullable = true)
 |-- sent_at: string (nullable = true)



In [0]:
from pyspark.sql.functions import expr, trim, to_timestamp, concat_ws, hour, dayofweek, when as w, lit, current_timestamp

numeric_cols = [
    "Global_active_power", "Global_reactive_power", "Voltage",
    "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"
]

df_typed = df_parsed
for c in numeric_cols:
    df_typed = df_typed.withColumn(c, expr(f"try_cast(trim({c}) as double)"))

df_features = (
    df_typed
    .withColumn("reading_timestamp", to_timestamp(concat_ws(" ", col("Date"), col("Time")), "d/M/yyyy H:mm:ss"))
    .withColumnRenamed("Global_active_power", "global_active_power_kw")
    .withColumnRenamed("Global_reactive_power", "global_reactive_power_kw")
    .withColumnRenamed("Voltage", "voltage_v")
    .withColumnRenamed("Global_intensity", "global_intensity_a")
    .withColumnRenamed("Sub_metering_1", "sub_metering_kitchen_wh")
    .withColumnRenamed("Sub_metering_2", "sub_metering_laundry_wh")
    .withColumnRenamed("Sub_metering_3", "sub_metering_waterheater_ac_wh")
    .withColumn(
        "unmetered_power_wh",
        (col("global_active_power_kw") * 1000 / 60)
        - (col("sub_metering_kitchen_wh") + col("sub_metering_laundry_wh") + col("sub_metering_waterheater_ac_wh"))
    )
    .withColumn("hour_of_day", hour(col("reading_timestamp")))
    .withColumn("is_weekend", dayofweek(col("reading_timestamp")).isin([1, 7]))
    .withColumn(
        "anomaly_type",
        w((col("voltage_v") < 220) | (col("voltage_v") > 256), lit("Voltage_Anomaly")).otherwise(lit(None))
    )
    .withColumn(
        "severity",
        w(col("anomaly_type").isNull(), lit(None))
        .when((col("voltage_v") < 220 * 0.85) | (col("voltage_v") > 256 * 1.2), lit("High"))
        .otherwise(lit("Medium"))
    )
    .withColumn("meter_id", lit("HOUSE_001"))
    .withColumn("ingestion_source", lit("Streaming"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
jdbc_hostname = "sql-segrid-anuj.database.windows.net"
jdbc_port = 1433
jdbc_database = "EnergyGridDB"
jdbc_username = "sqlamin"  
jdbc_password = "password"

jdbc_url = f"jdbc:sqlserver://{jdbc_hostname}:{jdbc_port};database={jdbc_database}"

connection_props = {
    "user": jdbc_username,
    "password": jdbc_password,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
def write_to_sql(batch_df, batch_id):
    if batch_df.isEmpty():          
        return

    fact_cols = [
        "meter_id", "reading_timestamp", "global_active_power_kw", "global_reactive_power_kw",
        "voltage_v", "global_intensity_a", "sub_metering_kitchen_wh", "sub_metering_laundry_wh",
        "sub_metering_waterheater_ac_wh", "unmetered_power_wh", "is_weekend",
        "ingestion_source", "ingestion_timestamp"
    ]
    batch_df.select(*fact_cols).write.mode("append").jdbc(
        url=jdbc_url, table="fact_energy_readings", properties=connection_props
    )

    anomalies = batch_df.filter(col("anomaly_type").isNotNull()).select(
        "reading_timestamp", "anomaly_type", "voltage_v",
        "global_reactive_power_kw", "global_intensity_a", "severity"
    )
    if not anomalies.isEmpty():      
        anomalies.write.mode("append").jdbc(
            url=jdbc_url, table="dbo.PowerQualityAnomalies", properties=connection_props
        )

    print(f"Batch {batch_id}: wrote {batch_df.count()} rows ({anomalies.count()} anomalies)")

In [0]:
print(repr(eh_namespace))

'evhns-segrid-anuj-std.servicebus.windows.net:9093'


In [0]:
%sh
nslookup evhs-segrid-anuj.servicebus.windows.net

Server:		168.63.129.16
Address:	168.63.129.16#53

Non-authoritative answer:
evhs-segrid-anuj.servicebus.windows.net	canonical name = ns-prod-pn1-510.centralindia.cloudapp.azure.com.
Name:	ns-prod-pn1-510.centralindia.cloudapp.azure.com
Address: 98.70.20.198



In [0]:
%sh
nc -zv evhs-segrid-anuj.servicebus.windows.net 9093

ns-prod-pn1-510.centralindia.cloudapp.azure.com [98.70.20.198] 9093 (?) open


In [0]:
query = (
    df_features.writeStream
    .foreachBatch(write_to_sql)
    .option("checkpointLocation", f"abfss://curated@{storage_account_name}.dfs.core.windows.net/_checkpoints/streaming_query/")
    .trigger(processingTime="30 seconds")
    .start()
)

query.awaitTermination()

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:132)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:132)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:129)
	at scala.collection.immutable.Range.foreach(Range.scala:190)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:129)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:721)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:441)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:441)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can